# Hugging Face Model Setup

In [ ]:
pip install transformers accelerate

To load the `google/gemma-2-2b-it` model, you might need to authenticate with Hugging Face. You can do this by logging in using your Hugging Face token. If you don't have one, you can create it on the Hugging Face website. Once you have the token, you can run the following code in a new cell or uncomment the line and replace `HF_TOKEN` with your actual token.

```python
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
```

In [ ]:
from huggingface_hub import login
login(token="YOUR_TOKEN_HERE")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "google/gemma-2-2b-it"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)

print(f"Model '{model_id}' and tokenizer loaded successfully.")

## Hugging Face Model: Text Generation (Gemma)

In [ ]:
input_text = "Explain the concept of inflation to a high school student."
input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate output
outputs = model.generate(**input_ids, max_new_tokens=200, num_return_sequences=1, do_sample=True, temperature=1.0)
response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the model's response if the input is echoed
if response_text.startswith(input_text):
    response_text = response_text[len(input_text):].strip()

print(response_text)

# Gemini API Setup

## Using the Gemini API

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in [Google AI Studio](https://makersuite.google.com/app/apikey).

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then, we can use it to configure the SDK.

In [ ]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GoogleAPIKey')
genai.configure(api_key=GOOGLE_API_KEY)

Now that the API is configured, we can initialize a generative model. I'll use `gemini-pro` as an example, but you can choose other models as well.

In [ ]:
# Initialize the Gemini API
gemini_model = genai.GenerativeModel('gemini-2.5-flash')

## Gemini API: Text Generation (Default Parameters)

With the model initialized, you can make API calls. Here's an example of generating text, such as a short story:

In [ ]:
# prompt_gemini = "Write a short story about a brave knight and a wise dragon."
prompt_gemini = "Explain the concept of inflation to a high school student."
gemini_response = gemini_model.generate_content(prompt_gemini)
print(gemini_response.text)

## Gemini API: Controlling Generation with Parameters (Temperature & Token Limit)

In [ ]:
from google.generativeai.types import GenerationConfig

# Define a new prompt for the demonstration
new_prompt_gemini = "Explain the concept of inflation to a high school student."
#"Write a short, creative haiku about a coding bug."

# Configure generation parameters
new_generation_config = GenerationConfig(
    temperature=0.9, # Higher temperature for more creative output
    max_output_tokens=2000 # Set a short token limit for a haiku
)

# Make a new API call with the specified parameters
new_gemini_response = gemini_model.generate_content(new_prompt_gemini, generation_config=new_generation_config)
print(new_gemini_response.text)

In [ ]:
from google.generativeai.types import GenerationConfig

# Define a new prompt for the demonstration
new_prompt_gemini = "Explain the concept of inflation to a high school student."
#"Write a short, creative haiku about a coding bug."

# Configure generation parameters
new_generation_config = GenerationConfig(
    temperature=0.3, # Higher temperature for more creative output
    max_output_tokens=2000 # Set a short token limit for a haiku
)

# Make a new API call with the specified parameters
new_gemini_response = gemini_model.generate_content(new_prompt_gemini, generation_config=new_generation_config)
print(new_gemini_response.text)

## Comparing Open-Source vs. Closed-Source Models

### Open-Source Models

**Definition:** Models whose code, weights, and sometimes data are freely available for anyone to inspect, modify, and use.

**Advantages:**

*   **Transparency:** The inner workings of the model are visible, allowing for better understanding, auditing, and debugging.
*   **Customization:** Users can fine-tune, modify, or extend the model to suit specific needs or datasets.
*   **Community Support:** Often benefit from a large, active community that contributes to development, documentation, and troubleshooting.
*   **Cost-Effective:** Typically free to use, though infrastructure costs for deployment still apply.
*   **Innovation:** Fosters rapid experimentation and innovation as researchers and developers can build upon existing models.

**Disadvantages:**

*   **Responsibility:** Users are responsible for their own security, deployment, and maintenance.
*   **Performance:** May not always match the cutting-edge performance of proprietary models from large companies, which often have access to vast computational resources and data.
*   **Complexity:** Can sometimes be more challenging to set up and manage without dedicated support.

### Closed-Source Models (Proprietary Models)

**Definition:** Models developed and maintained by private companies, where the code, weights, and architecture are not publicly accessible. Users typically access them via APIs.

**Advantages:**

*   **Ease of Use:** Often come with robust APIs, comprehensive documentation, and managed services, making them easier to integrate and deploy.
*   **Performance:** Companies like Google (Gemini), OpenAI (GPT), and Anthropic (Claude) often have proprietary models that are state-of-the-art in performance due to massive investments in data, compute, and research.
*   **Support:** Users typically receive professional support, maintenance, and updates from the provider.
*   **Reliability & Security:** Providers often ensure high availability, scalability, and security measures.

**Disadvantages:**

*   **Lack of Transparency:** The 'black box' nature means users cannot inspect the internal workings, which can be a concern for auditing, bias detection, or understanding failure modes.
*   **Vendor Lock-in:** Switching providers can be difficult due to API dependencies and different model behaviors.
*   **Cost:** Usage typically involves recurring costs based on API calls, token usage, or subscription fees.
*   **Limited Customization:** Fine-tuning options may be limited or non-existent, depending on the provider's offerings.
*   **Data Privacy:** Users must trust the provider with their data when using their APIs, raising privacy and compliance concerns for sensitive information.

### Conclusion

The choice between open-source and closed-source models depends on specific project requirements, budget, technical expertise, and the importance of factors like transparency, customization, performance, and support. Open-source models like Gemma offer flexibility and cost savings for those willing to manage the infrastructure, while closed-source models like Gemini provide convenience and cutting-edge performance through managed services.